# 01 — Data Preparation

Copies the Roboflow-exported datasets into a writable directory and fixes the
`data.yaml` paths (Roboflow exports use relative paths that break once the
data is moved).

Two datasets:
- **crop-data**: card boundary detection (`crn`, `nid`) — 2 classes
- **all-fields-dataset**: 15 field classes (front + back combined)

In [1]:
# Check GPU availability
import torch
print("GPU available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

GPU متاحة: True
اسم الـ GPU: Tesla T4


In [1]:
# List raw datasets as provided by Kaggle input
!ls /kaggle/input/datasets/stud20230837

all-fields-dataset  crop-data  test-cardss  test-imge


In [1]:
print("--- All Fields dataset ---")
!ls /kaggle/input/datasets/stud20230837/all-fields-dataset

print()
print("--- Crop Data dataset ---")
!ls /kaggle/input/datasets/stud20230837/crop-data/croping_data

--- All Fields ---
data.yaml  README.dataset.txt  README.roboflow.txt  test  train  valid

--- Crop Data ---
data.yaml  README.dataset.txt  README.roboflow.txt  test  train  valid


In [1]:
# Inspect the original data.yaml files (Roboflow uses relative paths: ../train/images)
print("--- All Fields data.yaml ---")
with open('/kaggle/input/datasets/stud20230837/all-fields-dataset/data.yaml', 'r') as f:
    print(f.read())

print("--- Crop Data data.yaml ---")
with open('/kaggle/input/datasets/stud20230837/crop-data/croping_data/data.yaml', 'r') as f:
    print(f.read())

--- All Fields data.yaml ---
train: ../train/images
val: ../valid/images
test: ../test/images

nc: 15
names: ['address', 'birth_date', 'country', 'doc_type', 'education', 'expire_date', 'gender', 'husband', 'image', 'issue_date', 'job', 'martial_state', 'name', 'national_id', 'religion']

roboflow:
  workspace: araid
  project: ara_id_processing-7sidx
  version: 4
  license: CC BY 4.0
  url: https://universe.roboflow.com/araid/ara_id_processing-7sidx/dataset/4

--- Crop Data data.yaml ---
train: ../train/images
val: ../valid/images
test: ../test/images

nc: 2
names: ['crn', 'nid']

roboflow:
  workspace: stud-workspace
  project: cards_detection-79ojy-i4p9w
  version: 1
  license: CC BY 4.0
  url: https://universe.roboflow.com/stud-workspace/cards_detection-79ojy-i4p9w/dataset/1


In [1]:
import shutil
import os

def safe_copytree(src, dst):
    """Copy a dataset folder, overwriting if it already exists.
    Needed because /kaggle/working persists across session restarts."""
    if os.path.exists(dst):
        shutil.rmtree(dst)
        print(f"Removed old: {dst}")
    shutil.copytree(src, dst)
    print(f"Copied: {dst}")

safe_copytree('/kaggle/input/datasets/stud20230837/all-fields-dataset', '/kaggle/working/all-fields-dataset')
safe_copytree('/kaggle/input/datasets/stud20230837/crop-data/croping_data', '/kaggle/working/crop-data')

تم مسح /kaggle/working/all-fields-dataset القديم
تم نسخ /kaggle/working/all-fields-dataset من جديد ✅
تم مسح /kaggle/working/crop-data القديم
تم نسخ /kaggle/working/crop-data من جديد ✅


In [1]:
# Rewrite data.yaml with absolute paths (train/valid/test sit next to data.yaml)
yaml_content_crop = """train: /kaggle/working/crop-data/train/images
val: /kaggle/working/crop-data/valid/images
test: /kaggle/working/crop-data/test/images

nc: 2
names: ['crn', 'nid']
"""
with open('/kaggle/working/crop-data/data.yaml', 'w') as f:
    f.write(yaml_content_crop)
print("Updated crop-data/data.yaml")

تم تعديل crop-data/data.yaml ✅


In [1]:
yaml_content_fields = """train: /kaggle/working/all-fields-dataset/train/images
val: /kaggle/working/all-fields-dataset/valid/images
test: /kaggle/working/all-fields-dataset/test/images

nc: 15
names: ['address', 'birth_date', 'country', 'doc_type', 'education', 'expire_date', 'gender',
        'husband', 'image', 'issue_date', 'job', 'martial_state', 'name', 'national_id', 'religion']
"""
with open('/kaggle/working/all-fields-dataset/data.yaml', 'w') as f:
    f.write(yaml_content_fields)
print("Updated all-fields-dataset/data.yaml")

تم تعديل all-fields-dataset/data.yaml ✅


In [ ]:
# Sanity check: confirm image counts after copy + yaml fix
import os

for path in [
    '/kaggle/working/all-fields-dataset/train/images',
    '/kaggle/working/all-fields-dataset/valid/images',
    '/kaggle/working/crop-data/train/images',
    '/kaggle/working/crop-data/valid/images',
]:
    count = len(os.listdir(path)) if os.path.exists(path) else 0
    print(f"{path} -> {count} images")